# 第 1 周末练习 —— 技术问答 / 游戏设计解释器

## 练习目标（理念）

为了展示你对 **OpenAI 兼容 API**（本练习经 **OpenRouter**）以及本地 **Ollama** 的熟悉程度，请构建一个小工具：

- **输入**：一个技术问题（本笔记本示例是「为独立开发者设计可落地的游戏方案」）
- **输出**：清晰、结构化的解释 / 设计说明
- **额外要求**：用**流式（streaming）**一边生成一边在笔记本里更新显示

这是你在课程期间也能自己天天用的工具：遇到想拆解的技术问题，丢进来分别问云端与本地模型。

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions API | `client.chat.completions.create(...)` |
| `messages`（system / user） | system 定「怎么答」，user 放具体问题 |
| 流式输出 `stream=True` | 逐块累加，并用 `update_display` 刷新 Markdown |
| 云端模型（经 OpenRouter） | `gpt-4o-mini`（常量 `MODEL_GPT`） |
| Ollama 本地模型 | `llama3.2`（常量 `MODEL_LLAMA`），OpenAI 兼容 `/v1` 基址 |

## 怎么跑

1. 从上到下依次运行每个单元格（Shift+Enter）
2. 准备好 `.env`：至少有 `OPENROUTER_API_KEY`；Ollama 路径还需要本机服务在跑，并已 `ollama pull llama3.2`
3. 在「提问」单元格改写 `question`（或改 `system_prompt`），再分别跑 GPT 与 Llama 两格做对比


In [1]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 导入标准库 os：读环境变量（Environment Variables），例如 API Key
import os
# 导入标准库 json：本练习主流程未强依赖，保留以便扩展解析结构化输出
import json
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 IPython.display 导入：Markdown 渲染、首次 display、流式 update_display
from IPython.display import Markdown, display, update_display
# 从 openai 导入 OpenAI 客户端：同时用于 OpenRouter 与 Ollama（都提供 OpenAI 兼容接口）
from openai import OpenAI


In [2]:
# ========== 常量：模型名与基址集中写在一处，后面只改这里 ==========

# 经 OpenRouter 调用的云端小模型：便宜、够用
MODEL_GPT = 'gpt-4o-mini'
# 本地 Ollama 模型名：需事先 ollama pull llama3.2；须与本机已安装名一致
MODEL_LLAMA = 'llama3.2'
# OpenRouter 的 OpenAI 兼容 API 基址（影响请求发往何处，URL 不翻译）
OPENROUTER_BASE_URL = 'https://openrouter.ai/api/v1'
# 本机 Ollama 的 OpenAI 兼容基址（默认 11434 端口的 /v1）
OLLAMA_BASE_URL = 'http://localhost:11434/v1'


In [3]:
# ========== 环境 + 双客户端：一条走 OpenRouter，一条走本地 Ollama ==========

# override=True：.env 中的值覆盖进程里已有同名变量
load_dotenv(override=True)
# 从环境变量读取 OpenRouter 密钥（键名必须与 .env 一致）
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
# Ollama 兼容接口也常需要 api_key 字段；本地常用占位/任意非空值，具体以你的 .env 为准
ollama_api_key = os.getenv("OLLAMA_API_KEY")

# 云端客户端：base_url 指到 OpenRouter，api_key 用 OpenRouter 的 Key
openai_client = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)

# 本地客户端：同一个 OpenAI SDK，只是 base_url 改成本机 Ollama
ollama_client = OpenAI(base_url=OLLAMA_BASE_URL, api_key=ollama_api_key)


In [4]:
# ========== System Prompt：定「独立游戏设计顾问」人设与输出结构 ==========

# 三引号内英文会原样发给模型——翻译会改变回答风格/结构，故保留英文
system_prompt = """
You are an expert indie game designer and technical game architect.

Your role is to generate practical, implementable game ideas for a solo developer.

Guidelines:
- Focus on core gameplay mechanics and game loops.
- Avoid vague descriptions.
- Keep art requirements minimal unless explicitly requested.
- Suggest systems that are technically feasible for a small team or solo developer.
- When appropriate, describe how the game could be structured in code (systems, state management, logic flow).
- Prioritize replayability and retention.
- Be concise but clear.
- Use structured sections when explaining ideas.

Always structure your answers like this:

1. Game Concept (1–2 sentences)
2. Core Loop
3. Core Mechanics
4. Progression System
5. Technical Implementation Notes
6. Why This Is Good For a Solo Developer
"""


In [7]:
# ========== 通用流式提问函数：同一套代码可喂给 OpenRouter 或 Ollama ==========

def ask_llm(question, client, model, system_prompt, temperature=0.7):
    """通用流式 LLM 调用：拼 messages → stream=True → 边收边刷新 Markdown → 返回全文。"""

    # messages：system 定规则，user 放具体问题（Chat Completions 标准格式）
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ]

    # stream=True：不要等整段生成完，而是持续返回增量 delta
    stream = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
        stream=True
    )

    # response：把各 chunk 拼成完整字符串，最后 return 给调用方
    response = ""
    # display_id=True：拿到可更新的显示句柄，后面用 update_display 原地刷新
    display_handle = display(Markdown(""), display_id=True)

    # 遍历流式 chunk；有的 chunk 只有元数据、没有 content，需要判空
    for chunk in stream:
        if chunk.choices[0].delta.content:
            # 累加本块文本
            response += chunk.choices[0].delta.content
            # 用同一 display_id 刷新，实现「打字机」效果
            update_display(Markdown(response), display_id=display_handle.display_id)

    # 返回完整回答文本，便于赋值到变量或后续处理
    return response


In [5]:
# ========== 提问：改这里的字符串就能问新问题 ==========

# 发给模型的用户问题保持英文（影响回答的字符串不翻译）
# 练习建议：换成你自己的技术问题，再分别跑下面 GPT / Llama 两格对比
question = """
Design a single-player game that emphasizes strong gameplay systems over heavy art requirements.

Constraints:
- Must be buildable by one experienced developer in 6–8 weeks.
- Minimal art assets.
- High replayability.
- No multiplayer.

Explain:
- The core game concept
- Core gameplay loop
- Main systems and how they interact
- Progression and replayability mechanics
- Technical implementation considerations
"""


In [ ]:
# ========== 路径 A：用云端 gpt-4o-mini（经 OpenRouter）流式回答 ==========

# ask_llm 内部会 stream + update_display；返回值是完整字符串
gpt_response = ask_llm(
    question=question,
    client=openai_client,
    model=MODEL_GPT,
    system_prompt=system_prompt
)


In [ ]:
# ========== 路径 B：用本地 Llama 3.2（Ollama）流式回答 ==========

# 复用同一 ask_llm；只把 client / model 换成本地后端
# 理念：同一问题、两个后端 —— 对比云端与本地的速度、风格、是否需要付费 Key
# 前提：Ollama 在跑，且已安装 llama3.2
llama_response = ask_llm(
    question=question,
    client=ollama_client,
    model=MODEL_LLAMA,
    system_prompt=system_prompt
)
